# **Langkah 2: Melatih Model Baseline (8 Kategori) Multiclass**

###**2.1 Definisi Fungsi Pembantu (Helper Functions)**

In [ ]:
def get_memory_usage():
    """Mendapatkan penggunaan RAM sistem"""
    memory = psutil.virtual_memory()
    used_gb = (memory.total - memory.available) / (1024**3)
    total_gb = memory.total / (1024**3)
    available_gb = memory.available / (1024**3)
    percent = memory.percent
    return used_gb, total_gb, available_gb, percent

def print_memory_status(stage=""):
    """Print status memori"""
    used, total, available, percent = get_memory_usage()
    print(f"\n[Memory Status - {stage}]")
    print(f"  Used: {used:.2f} GB / {total:.2f} GB ({percent:.1f}%)")
    print(f"  Available: {available:.2f} GB")

def clear_memory():
    """Bersihkan memory"""
    gc.collect()
    print("  Memory cleared")

def print_detailed_report(y_true, y_pred, model_name, label_encoder):
    """Print detailed classification report per category"""
    from sklearn.metrics import classification_report, accuracy_score

    print("\n" + "="*80)
    print(f"DETAILED CLASSIFICATION REPORT: {model_name}")
    print("="*80)

    # Get category names
    target_names = label_encoder.classes_

    # Generate classification report
    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        digits=4,
        zero_division=0
    )

    print("\n" + report)

    # Calculate overall accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f"\nConfirmed Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

    # Parse report to dictionary
    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )

    # Create detailed table view
    print("\n" + "="*80)
    print(f"PER-CATEGORY METRICS (Table Format)")
    print("="*80)
    print(f"{'Category':<20} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
    print("-"*80)

    # Print each category
    for category in target_names:
        if category in report_dict:
            p = report_dict[category]['precision']
            r = report_dict[category]['recall']
            f1 = report_dict[category]['f1-score']
            support = int(report_dict[category]['support'])

            print(f"{category:<20} {p:>10.4f} {r:>10.4f} {f1:>10.4f} {support:>10,}")

    print("-"*80)

    # Print averages
    print(f"{'Macro Avg':<20} {report_dict['macro avg']['precision']:>10.4f} "
          f"{report_dict['macro avg']['recall']:>10.4f} "
          f"{report_dict['macro avg']['f1-score']:>10.4f} "
          f"{int(report_dict['macro avg']['support']):>10,}")

    print(f"{'Weighted Avg':<20} {report_dict['weighted avg']['precision']:>10.4f} "
          f"{report_dict['weighted avg']['recall']:>10.4f} "
          f"{report_dict['weighted avg']['f1-score']:>10.4f} "
          f"{int(report_dict['weighted avg']['support']):>10,}")

    print("="*80)

    return report_dict

###**2.2 Load Kembali Hasil Pre-Processing Data**

In [ ]:
# Definisi Path
PROCESSED_DIR = '/content/drive/My Drive/Baseline/Processed_Data/'
MODELS_DIR = '/content/drive/My Drive/Baseline/Models/'
RESULTS_DIR = '/content/drive/My Drive/Baseline/Results/'
CHECKPOINT_DIR = '/content/drive/My Drive/Baseline/Checkpoints/'

# Direktori/folder
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print_memory_status("Setelah Mount Drive")

print("\nMemuat data terproses dari Google Drive...")

# Load dengan nama file baru
X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))
y_test = np.load(os.path.join(PROCESSED_DIR, 'y_test.npy'))

# UBAH: Load label encoder untuk kategori
with open(os.path.join(PROCESSED_DIR, 'label_encoder_category.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)

with open(os.path.join(PROCESSED_DIR, 'feature_names.json'), 'r') as f:
    feature_names = json.load(f)

print(f"\nData dimuat:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Jumlah kategori: {len(np.unique(y_train))}") # UBAH: kelas -> kategori
print(f"Jumlah fitur: {len(feature_names)}")

print_memory_status("Setelah Load Data")


[Memory Status - Setelah Mount Drive]
  Used: 2.13 GB / 50.99 GB (4.2%)
  Available: 48.86 GB

Memuat data terproses dari Google Drive...

Data dimuat:
X_train shape: (17118883, 39)
X_test shape: (4279721, 39)
y_train shape: (17118883,)
y_test shape: (4279721,)
Jumlah kategori: 8
Jumlah fitur: 39

[Memory Status - Setelah Load Data]
  Used: 8.49 GB / 50.99 GB (16.7%)
  Available: 42.50 GB


###**2.3 Definisi Fungsi Evaluasi**

In [ ]:
def evaluate_model(y_true, y_pred, model_name, training_time):
    """
    Fungsi untuk evaluasi model dengan berbagai metrik
    """
    # Calculate all metrics
    results = {
        'model_name': model_name,
        'dataset_type': 'Full',
        'training_time_seconds': training_time,

        # Accuracy
        'accuracy': accuracy_score(y_true, y_pred),

        # Precision (3 averaging methods)
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),

        # Recall (3 averaging methods)
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),

        # F1-Score (3 averaging methods)
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),

        # Matthews Correlation Coefficient
        'mcc': matthews_corrcoef(y_true, y_pred)
    }

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    # Classification Report (detail per class)
    class_names = label_encoder.classes_
    report = classification_report(
        y_true, y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    return results, cm, report

In [ ]:
def print_results(results):
    """
    Print hasil evaluasi dengan format yang rapi
    """
    print(f"\n{'='*60}")
    print(f"HASIL EVALUASI: {results['model_name']} ({results['dataset_type']})")
    print(f"{'='*60}")
    print(f"Waktu Training: {results['training_time_seconds']:.2f} detik ({results['training_time_seconds']/60:.2f} menit)")

    print(f"\nMetrik Utama:")
    print(f"  Accuracy           : {results['accuracy']:.4f}")

    print(f"\nPrecision:")
    print(f"  Macro              : {results['precision_macro']:.4f}")
    print(f"  Weighted           : {results['precision_weighted']:.4f}")
    print(f"  Micro              : {results['precision_micro']:.4f}")

    print(f"\nRecall:")
    print(f"  Macro              : {results['recall_macro']:.4f}")
    print(f"  Weighted           : {results['recall_weighted']:.4f}")
    print(f"  Micro              : {results['recall_micro']:.4f}")

    print(f"\nF1-Score:")
    print(f"  Macro              : {results['f1_macro']:.4f}")
    print(f"  Weighted           : {results['f1_weighted']:.4f}")
    print(f"  Micro              : {results['f1_micro']:.4f}")

    print(f"\nMCC                 : {results['mcc']:.4f}")
    print(f"{'='*60}")

In [ ]:
all_results = []

###**2.4 Training Random Forest**

In [ ]:
start_time = time.time()

rf_full = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    class_weight=None,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Mulai training...")
rf_full.fit(X_train, y_train)
rf_full_time = time.time() - start_time

print("Melakukan prediksi...")
y_pred_rf_full = rf_full.predict(X_test)

# Evaluasi standar
rf_full_results, rf_full_cm, rf_full_report = evaluate_model(
    y_test, y_pred_rf_full, "Random Forest", rf_full_time
)
print_results(rf_full_results)

# Print detailed report per category
rf_detailed = print_detailed_report(y_test, y_pred_rf_full, "Random Forest", label_encoder)

all_results.append(rf_full_results)

# Simpan model
with open(os.path.join(MODELS_DIR, 'baseline_rf_category.pkl'), 'wb') as f:
    pickle.dump(rf_full, f)
np.save(os.path.join(RESULTS_DIR, 'cm_rf_category.npy'), rf_full_cm)
with open(os.path.join(RESULTS_DIR, 'report_rf_category.json'), 'w') as f:
    json.dump(rf_full_report, f, indent=4)

del rf_full
clear_memory()
print_memory_status("Setelah Training RF")

Mulai training...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  7.6min
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed: 36.6min
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed: 59.6min finished


Melakukan prediksi...


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:   11.8s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:   54.7s
[Parallel(n_jobs=8)]: Done 300 out of 300 | elapsed:  1.5min finished



HASIL EVALUASI: Random Forest (Full)
Waktu Training: 3581.37 detik (59.69 menit)

Metrik Utama:
  Accuracy           : 0.8562

Precision:
  Macro              : 0.8728
  Weighted           : 0.8502
  Micro              : 0.8562

Recall:
  Macro              : 0.6678
  Weighted           : 0.8562
  Micro              : 0.8562

F1-Score:
  Macro              : 0.7075
  Weighted           : 0.8461
  Micro              : 0.8562

MCC                 : 0.7560

DETAILED CLASSIFICATION REPORT: Random Forest

              precision    recall  f1-score   support

      Benign     0.8282    0.8976    0.8615    218760
 Brute_Force     0.9314    0.2236    0.3606      2612
        DDoS     0.8532    0.9480    0.8981   2520784
         DoS     0.7552    0.4961    0.5988    813818
       Mirai     0.9995    0.9976    0.9986    490925
       Recon     0.7895    0.7744    0.7819    136894
    Spoofing     0.9524    0.8517    0.8992     90981
   Web_Based     0.8734    0.1534    0.2610      4947

    a

###**2.5 Training XGBoost**

In [ ]:
start_time = time.time()

xgb_full = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0,
    reg_alpha=0.1,
    reg_lambda=1,
    objective='multi:softmax',
    num_class=len(np.unique(y_train)), # Otomatis jadi 8
    eval_metric='mlogloss',
    tree_method='hist', # Mengubah 'gpu_hist' menjadi 'hist' karena error menunjukkan 'gpu_hist' tidak dikenali
    device='cuda', # Tetap menggunakan 'cuda' jika runtime mendukung, namun 'tree_method' yang penting
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

In [ ]:
print("Mulai training...")
xgb_full.fit(X_train,
             y_train,
             verbose=True)
xgb_full_time = time.time() - start_time

print("Melakukan prediksi...")
y_pred_xgb_full = xgb_full.predict(X_test)

# Evaluasi standar
xgb_full_results, xgb_full_cm, xgb_full_report = evaluate_model(
    y_test, y_pred_xgb_full, "XGBoost", xgb_full_time
)
print_results(xgb_full_results)

# Print detailed report per category
xgb_detailed = print_detailed_report(y_test, y_pred_xgb_full, "XGBoost", label_encoder)

all_results.append(xgb_full_results)

# Simpan model
xgb_full.save_model(os.path.join(MODELS_DIR, 'baseline_xgboost_category.json'))
np.save(os.path.join(RESULTS_DIR, 'cm_xgboost_category.npy'), xgb_full_cm)
with open(os.path.join(RESULTS_DIR, 'report_xgboost_category.json'), 'w') as f:
    json.dump(xgb_full_report, f, indent=4)

del xgb_full
clear_memory()
print_memory_status("Setelah Training XGBoost")

Mulai training...
Melakukan prediksi...

HASIL EVALUASI: XGBoost (Full)
Waktu Training: 413.33 detik (6.89 menit)

Metrik Utama:
  Accuracy           : 0.8538

Precision:
  Macro              : 0.8542
  Weighted           : 0.8472
  Micro              : 0.8538

Recall:
  Macro              : 0.6747
  Weighted           : 0.8538
  Micro              : 0.8538

F1-Score:
  Macro              : 0.7155
  Weighted           : 0.8436
  Micro              : 0.8538

MCC                 : 0.7518

DETAILED CLASSIFICATION REPORT: XGBoost

              precision    recall  f1-score   support

      Benign     0.8243    0.8957    0.8585    218760
 Brute_Force     0.8519    0.2730    0.4135      2612
        DDoS     0.8518    0.9460    0.8964   2520784
         DoS     0.7462    0.4911    0.5923    813818
       Mirai     0.9992    0.9980    0.9986    490925
       Recon     0.7899    0.7670    0.7783    136894
    Spoofing     0.9485    0.8516    0.8974     90981
   Web_Based     0.8218    0.1753 

###**2.7 Training LightGBM (CPU Versio)**

In [ ]:
print("Catatan: Menggunakan LightGBM CPU untuk stabilitas memori")

# Clear memory sebelum training
clear_memory()
print_memory_status("\nSebelum Training LightGBM")

Catatan: Menggunakan LightGBM CPU untuk stabilitas memori
  Memory cleared

[Memory Status - 
Sebelum Training LightGBM]
  Used: 8.85 GB / 50.99 GB (17.3%)
  Available: 42.14 GB


In [ ]:
start_time = time.time()

# Callback untuk logging progress
class LightGBMCallback:
    def __init__(self, total_iterations):
        self.total_iterations = total_iterations
        self.current_iteration = 0
        self.start_time = time.time()

    def __call__(self, env):
        self.current_iteration += 1
        if self.current_iteration % 10 == 0 or self.current_iteration == self.total_iterations:
            progress = (self.current_iteration / self.total_iterations) * 100
            elapsed = time.time() - self.start_time
            used, total, available, percent = get_memory_usage()
            print(f"  Iteration {self.current_iteration}/{self.total_iterations} "
                  f"({progress:.1f}%) - "
                  f"Time: {elapsed:.1f}s - "
                  f"RAM: {percent:.1f}% used ({available:.1f} GB available)")

lgb_callback = LightGBMCallback(100)

In [ ]:
lgb_full = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.1,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    objective='multiclass',
    num_class=len(np.unique(y_train)), # Otomatis jadi 8
    metric='multi_logloss',
    class_weight= None,
    device='cpu',  # Menggunakan CPU untuk stabilitas
    random_state=42,
    n_jobs=-1,
    verbose=-1  # Suppress default logging
)

In [ ]:
print("Mulai training LightGBM...")
print("\nProgress akan ditampilkan setiap 10 iterasi")

try:
    lgb_full.fit(
        X_train, y_train,
        callbacks=[lgb_callback]
    )

    lgb_full_time = time.time() - start_time
    print(f"\nTraining selesai dalam {lgb_full_time:.2f} detik ({lgb_full_time/60:.2f} menit)")

    print_memory_status("Setelah Training LightGBM")

    # Prediksi
    print("\nMelakukan prediksi pada test set...")
    y_pred_lgb_full = lgb_full.predict(X_test)

    # Evaluasi Standar
    lgb_full_results, lgb_full_cm, lgb_full_report = evaluate_model(
        y_test, y_pred_lgb_full, "LightGBM", lgb_full_time
    )
    print_results(lgb_full_results)

    # Print detailed report per category
    lgb_detailed = print_detailed_report(y_test, y_pred_lgb_full, "LightGBM", label_encoder)

    all_results.append(lgb_full_results)

    # Simpan model
    print("\nMenyimpan model LightGBM...")
    with open(os.path.join(MODELS_DIR, 'baseline_lightgbm_category.pkl'), 'wb') as f:
        pickle.dump(lgb_full, f)
    np.save(os.path.join(RESULTS_DIR, 'cm_lightgbm_category.npy'), lgb_full_cm)
    with open(os.path.join(RESULTS_DIR, 'report_lightgbm_category.json'), 'w') as f:
        json.dump(lgb_full_report, f, indent=4)
    print("Model LightGBM tersimpan!")

    # Clear memory
    del y_pred_lgb_full
    clear_memory()

except Exception as e:
    print(f"\nError saat training LightGBM: {e}")
    print("Melanjutkan ke training DNN...")

print_memory_status("Setelah LightGBM")

Mulai training LightGBM...

Progress akan ditampilkan setiap 10 iterasi
  Iteration 10/100 (10.0%) - Time: 60.0s - RAM: 26.1% used (37.7 GB available)
  Iteration 20/100 (20.0%) - Time: 98.0s - RAM: 26.1% used (37.7 GB available)
  Iteration 30/100 (30.0%) - Time: 137.5s - RAM: 26.1% used (37.7 GB available)
  Iteration 40/100 (40.0%) - Time: 177.7s - RAM: 26.1% used (37.7 GB available)
  Iteration 50/100 (50.0%) - Time: 218.1s - RAM: 26.1% used (37.7 GB available)
  Iteration 60/100 (60.0%) - Time: 258.7s - RAM: 26.1% used (37.7 GB available)
  Iteration 70/100 (70.0%) - Time: 300.6s - RAM: 26.1% used (37.7 GB available)
  Iteration 80/100 (80.0%) - Time: 343.4s - RAM: 26.1% used (37.7 GB available)
  Iteration 90/100 (90.0%) - Time: 388.5s - RAM: 26.1% used (37.7 GB available)
  Iteration 100/100 (100.0%) - Time: 433.4s - RAM: 26.1% used (37.7 GB available)
  Iteration 110/100 (110.0%) - Time: 478.2s - RAM: 26.1% used (37.7 GB available)
  Iteration 120/100 (120.0%) - Time: 526.4s - 

###**2.8 Training Deep Neural Network (DNN)**

In [ ]:
# Clear memory sebelum training
clear_memory()
print_memory_status("Sebelum Training DNN")

  Memory cleared

[Memory Status - Sebelum Training DNN]
  Used: 9.86 GB / 50.99 GB (19.3%)
  Available: 41.13 GB


In [ ]:
num_features = X_train.shape[1]
num_classes = len(np.unique(y_train))

def create_dnn_model(num_features, num_classes):
    """
    Membuat arsitektur DNN untuk 8 Kategori
    """
    model = keras.Sequential([
        layers.Input(shape=(num_features,)),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.2),
        layers.BatchNormalization(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.2),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.BatchNormalization(),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax') # num_classes akan = 8
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Buat model dengan jumlah kelas yang benar
num_classes = len(np.unique(y_train))  # = 8

dnn_full = create_dnn_model(num_features, num_classes)

In [ ]:
# Custom callback untuk logging memory
class MemoryLoggingCallback(keras.callbacks.Callback):
    def __init__(self, log_frequency=1):
        super().__init__()
        self.log_frequency = log_frequency
        self.start_time = None

    def on_train_begin(self, logs=None):
        self.start_time = time.time()
        print("Training dimulai...")
        print_memory_status("Train Begin")

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.log_frequency == 0:
            elapsed = time.time() - self.start_time
            used, total, available, percent = get_memory_usage()
            print(f"\nEpoch {epoch + 1}:")
            print(f"  Loss: {logs.get('loss'):.4f} - Acc: {logs.get('accuracy'):.4f}")
            print(f"  Val Loss: {logs.get('val_loss'):.4f} - Val Acc: {logs.get('val_accuracy'):.4f}")
            print(f"  Time: {elapsed:.1f}s - RAM: {percent:.1f}% ({available:.1f} GB available)")

dnn_full = create_dnn_model(num_features, num_classes)

In [ ]:
print("Arsitektur DNN:")
dnn_full.summary()

# Callbacks
checkpoint_full = ModelCheckpoint(
    os.path.join(CHECKPOINT_DIR, 'dnn_full_best.h5'),
    monitor='val_loss',
    save_best_only=True,
    verbose=0
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1
)

memory_logger = MemoryLoggingCallback(log_frequency=1)

Arsitektur DNN:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 512)            │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 197,064 (769.78 KB)

 Trainable params: 195,272 (762.78 KB)

 Non-trainable params: 1,792 (7.00 KB)

In [ ]:
print("Mulai training DNN...")
start_time = time.time()

try:
    history_full = dnn_full.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=100,
        batch_size=4096,  # Batch size lebih besar untuk efisiensi memori (Sebelumnya 256)
        callbacks=[checkpoint_full, early_stopping, reduce_lr, memory_logger],
        verbose=1
    )

    dnn_full_time = time.time() - start_time
    print(f"\nTraining selesai dalam {dnn_full_time:.2f} detik ({dnn_full_time/60:.2f} menit)")

    print_memory_status("Setelah Training DNN")

    # Prediksi
    print("\nMelakukan prediksi pada test set...")
    y_pred_dnn_full_proba = dnn_full.predict(X_test, batch_size=512, verbose=0)
    y_pred_dnn_full = np.argmax(y_pred_dnn_full_proba, axis=1)

    # Evaluasi Standar
    dnn_full_results, dnn_full_cm, dnn_full_report = evaluate_model(
        y_test, y_pred_dnn_full, "Deep Neural Network", dnn_full_time
    )
    print_results(dnn_full_results)

    # Print detailed report per category
    dnn_detailed = print_detailed_report(y_test, y_pred_dnn_full, "Deep Neural Network", label_encoder)

    all_results.append(dnn_full_results)

except Exception as e:
    print(f"\nError saat training DNN: {e}")
    print("Jika error memory, coba kurangi batch_size atau epochs")

print_memory_status("Setelah DNN")

Mulai training DNN...
Training dimulai...

[Memory Status - Train Begin]
  Used: 17.50 GB / 50.99 GB (34.3%)
  Available: 33.49 GB
Epoch 1/100
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8141 - loss: 0.3871


Epoch 1:
  Loss: 0.3527 - Acc: 0.8271
  Val Loss: 0.3319 - Val Acc: 0.8369
  Time: 34.3s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 34s 7ms/step - accuracy: 0.8271 - loss: 0.3527 - val_accuracy: 0.8369 - val_loss: 0.3319 - learning_rate: 0.0010
Epoch 2/100
3761/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8356 - loss: 0.3340


Epoch 2:
  Loss: 0.3321 - Acc: 0.8364
  Val Loss: 0.3244 - Val Acc: 0.8397
  Time: 54.5s - RAM: 29.5% (35.9 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8364 - loss: 0.3321 - val_accuracy: 0.8397 - val_loss: 0.3244 - learning_rate: 0.0010
Epoch 3/100
3751/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8379 - loss: 0.3290
Epoch 3:
  Loss: 0.3282 - Acc: 0.8382
  Val Loss: 0.3249 - Val Acc: 0.8391
  Time: 74.0s - RAM: 29.7% (35.9 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.8382 - loss: 0.3282 - val_accuracy: 0.8391 - val_loss: 0.3249 - learning_rate: 0.0010
Epoch 4/100
3760/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8390 - loss: 0.3265


Epoch 4:
  Loss: 0.3257 - Acc: 0.8394
  Val Loss: 0.3213 - Val Acc: 0.8420
  Time: 94.2s - RAM: 29.8% (35.8 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8394 - loss: 0.3257 - val_accuracy: 0.8420 - val_loss: 0.3213 - learning_rate: 0.0010
Epoch 5/100
3753/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8402 - loss: 0.3242
Epoch 5:
  Loss: 0.3240 - Acc: 0.8402
  Val Loss: 0.3219 - Val Acc: 0.8417
  Time: 113.7s - RAM: 29.9% (35.7 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.8402 - loss: 0.3240 - val_accuracy: 0.8417 - val_loss: 0.3219 - learning_rate: 0.0010
Epoch 6/100
3751/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8405 - loss: 0.3235


Epoch 6:
  Loss: 0.3229 - Acc: 0.8407
  Val Loss: 0.3199 - Val Acc: 0.8426
  Time: 133.7s - RAM: 30.0% (35.7 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8407 - loss: 0.3229 - val_accuracy: 0.8426 - val_loss: 0.3199 - learning_rate: 0.0010
Epoch 7/100
3761/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8410 - loss: 0.3222

...


Epoch 74:
  Loss: 0.3074 - Acc: 0.8485
  Val Loss: 0.3058 - Val Acc: 0.8499
  Time: 1482.4s - RAM: 29.3% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.8485 - loss: 0.3074 - val_accuracy: 0.8499 - val_loss: 0.3058 - learning_rate: 1.2500e-04
Epoch 75/100
3753/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8485 - loss: 0.3075
Epoch 75:
  Loss: 0.3074 - Acc: 0.8485
  Val Loss: 0.3061 - Val Acc: 0.8498
  Time: 1502.3s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8485 - loss: 0.3074 - val_accuracy: 0.8498 - val_loss: 0.3061 - learning_rate: 1.2500e-04
Epoch 76/100
3759/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8485 - loss: 0.3075
Epoch 76:
  Loss: 0.3074 - Acc: 0.8485
  Val Loss: 0.3059 - Val Acc: 0.8500
  Time: 1522.3s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8485 - loss: 0.3074 - val_accuracy: 0.8500 - val_loss: 0.3059 - learning_rate: 1.2500e-04



Epoch 77:
  Loss: 0.3074 - Acc: 0.8485
  Val Loss: 0.3056 - Val Acc: 0.8501
  Time: 1543.3s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.8485 - loss: 0.3074 - val_accuracy: 0.8501 - val_loss: 0.3056 - learning_rate: 1.2500e-04
Epoch 78/100
3759/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8486 - loss: 0.3073
Epoch 78:
  Loss: 0.3074 - Acc: 0.8486
  Val Loss: 0.3064 - Val Acc: 0.8496
  Time: 1563.2s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8486 - loss: 0.3074 - val_accuracy: 0.8496 - val_loss: 0.3064 - learning_rate: 1.2500e-04
Epoch 79/100
3761/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8487 - loss: 0.3073
Epoch 79:
  Loss: 0.3074 - Acc: 0.8486
  Val Loss: 0.3064 - Val Acc: 0.8496
  Time: 1583.1s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8486 - loss: 0.3074 - val_accuracy: 0.8496 - val_loss: 0.3064 - learning_rate: 1.2500e-04



Epoch 80:
  Loss: 0.3073 - Acc: 0.8486
  Val Loss: 0.3054 - Val Acc: 0.8502
  Time: 1603.7s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.8486 - loss: 0.3073 - val_accuracy: 0.8502 - val_loss: 0.3054 - learning_rate: 1.2500e-04
Epoch 81/100
3753/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8485 - loss: 0.3075
Epoch 81:
  Loss: 0.3073 - Acc: 0.8486
  Val Loss: 0.3055 - Val Acc: 0.8502
  Time: 1623.7s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8486 - loss: 0.3073 - val_accuracy: 0.8502 - val_loss: 0.3055 - learning_rate: 1.2500e-04
Epoch 82/100
3757/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8487 - loss: 0.3073
Epoch 82:
  Loss: 0.3073 - Acc: 0.8486
  Val Loss: 0.3072 - Val Acc: 0.8497
  Time: 1643.6s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8486 - loss: 0.3073 - val_accuracy: 0.8497 - val_loss: 0.3072 - learning_rate: 1.2500e-04



Epoch 90:
  Loss: 0.3068 - Acc: 0.8490
  Val Loss: 0.3053 - Val Acc: 0.8503
  Time: 1804.0s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.8490 - loss: 0.3068 - val_accuracy: 0.8503 - val_loss: 0.3053 - learning_rate: 6.2500e-05
Epoch 91/100
3752/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8488 - loss: 0.3068
Epoch 91:
  Loss: 0.3068 - Acc: 0.8488
  Val Loss: 0.3058 - Val Acc: 0.8502
  Time: 1823.9s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8488 - loss: 0.3068 - val_accuracy: 0.8502 - val_loss: 0.3058 - learning_rate: 6.2500e-05
Epoch 92/100
3756/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8489 - loss: 0.3067
Epoch 92:
  Loss: 0.3068 - Acc: 0.8489
  Val Loss: 0.3058 - Val Acc: 0.8501
  Time: 1843.7s - RAM: 29.5% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8489 - loss: 0.3068 - val_accuracy: 0.8501 - val_loss: 0.3058 - learning_rate: 6.2500e-05



Epoch 96:
  Loss: 0.3065 - Acc: 0.8490
  Val Loss: 0.3052 - Val Acc: 0.8505
  Time: 1923.9s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.8490 - loss: 0.3065 - val_accuracy: 0.8505 - val_loss: 0.3052 - learning_rate: 3.1250e-05
Epoch 97/100
3752/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8490 - loss: 0.3067
Epoch 97:
  Loss: 0.3065 - Acc: 0.8490
  Val Loss: 0.3054 - Val Acc: 0.8503
  Time: 1943.7s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8490 - loss: 0.3065 - val_accuracy: 0.8503 - val_loss: 0.3054 - learning_rate: 3.1250e-05
Epoch 98/100
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8490 - loss: 0.3066
Epoch 98:
  Loss: 0.3065 - Acc: 0.8491
  Val Loss: 0.3056 - Val Acc: 0.8501
  Time: 1963.6s - RAM: 29.4% (36.0 GB available)
3762/3762 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8491 - loss: 0.3065 - val_accuracy: 0.8501 - val_loss: 0.3056 - learning_rate: 3.1250e-05


In [ ]:
# Simpan model dan history
print("\nMenyimpan model DNN...")
dnn_full.save(os.path.join(MODELS_DIR, 'baseline_dnn_category.h5'))  # UBAH
np.save(os.path.join(RESULTS_DIR, 'cm_dnn_category.npy'), dnn_full_cm)  # UBAH
with open(os.path.join(RESULTS_DIR, 'report_dnn_category.json'), 'w') as f:  # UBAH
    json.dump(dnn_full_report, f, indent=4)

history_full_dict = {
    'loss': [float(x) for x in history_full.history['loss']],
    'accuracy': [float(x) for x in history_full.history['accuracy']],
    'val_loss': [float(x) for x in history_full.history['val_loss']],
    'val_accuracy': [float(x) for x in history_full.history['val_accuracy']]
    }
with open(os.path.join(RESULTS_DIR, 'dnn_full_history.json'), 'w') as f:
    json.dump(history_full_dict, f, indent=4)

print("Model DNN tersimpan!")


Menyimpan model DNN...
Model DNN tersimpan!


In [ ]:
# Clear memory
del y_pred_dnn_full_proba, y_pred_dnn_full
clear_memory()

  Memory cleared


###**2.9 Perbandingan Hasil Semua Model Baseline**

In [ ]:
print("\n" + "="*70)
print("COMPREHENSIVE METRICS COMPARISON")
print("="*70)

comparison_df_fixed = pd.DataFrame(all_results)
print("\n", comparison_df_fixed.to_string(index=False))

# Save comparison
comparison_df_fixed.to_csv(
    os.path.join(RESULTS_DIR, 'baseline_comparison_category.csv'),
    index=False
)

# Verify no NaN values
print("\n" + "="*70)
print("VERIFICATION: Checking for NaN values...")
print("="*70)

nan_counts = comparison_df_fixed.isnull().sum()
if nan_counts.sum() == 0:
    print(" SUCCESS: No NaN values found!")
else:
    print(" WARNING: NaN values detected:")
    print(nan_counts[nan_counts > 0])

# Print best models
print("\n" + "="*70)
print("MODEL TERBAIK PER METRIK (UPDATED)")
print("="*70)

metrics_to_check = {
    'Accuracy': 'accuracy',
    'F1-Score (Macro)': 'f1_macro',
    'F1-Score (Micro)': 'f1_micro',
    'Recall (Macro)': 'recall_macro',
    'MCC': 'mcc'
}

for metric_name, metric_col in metrics_to_check.items():
    best_idx = comparison_df_fixed[metric_col].idxmax()
    best_row = comparison_df_fixed.loc[best_idx]
    print(f"\n{metric_name}:")
    print(f"  Model: {best_row['model_name']}")
    print(f"  Score: {best_row[metric_col]:.4f}")
    print(f"  Training Time: {best_row['training_time_seconds']:.2f}s")

print("\n" + "="*70)
print(" RE-EVALUATION COMPLETE!")
print("="*70)
print(f"\nUpdated results saved to:")
print(f"  - {os.path.join(RESULTS_DIR, 'baseline_comparison_category.csv')}")
print("="*70)



COMPREHENSIVE METRICS COMPARISON

          model_name dataset_type  training_time_seconds  accuracy  precision_macro  precision_weighted  precision_micro  recall_macro  recall_weighted  recall_micro  f1_macro  f1_weighted  f1_micro      mcc
            XGBoost         Full             413.332219  0.853839         0.854191            0.847242         0.853839      0.674703         0.853839      0.853839  0.715497     0.843618  0.853839 0.751783
           LightGBM         Full            2165.656919  0.857377         0.852160            0.851061         0.857377      0.687038         0.857377      0.857377  0.728680     0.848017  0.857377 0.757893
Deep Neural Network         Full            2010.907086  0.850418         0.862860            0.843430         0.850418      0.663124         0.850418      0.850418  0.702060     0.839321  0.850418 0.745867

VERIFICATION: Checking for NaN values...
 SUCCESS: No NaN values found!

MODEL TERBAIK PER METRIK (UPDATED)

Accuracy:
  Model: LightGB